# KVAE — Results
Evaluation and visualization for all three models: **KVAE**, **GRU-VAE**, **CV-VAE**.

## 1. Setup

In [ ]:
import sys, os
sys.path.append('..')

import torch
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from torch.utils.data import DataLoader

from config.vae_config import VAEConfig
from config.simulation_config import SimulationConfig
from config.train_config import TrainConfig
from models.kvae import KVAE
from models.gru_vae import GRUVAE
from models.cv_vae import CVVAE
from dataset.dataset import BallDataset
from evaluation.evaluate import load_model, compute_metrics, compute_mse_per_step
from utils.visualize import (
    plot_trajectories,
    plot_reconstruction_grid,
    plot_alpha,
    plot_uncertainty, 
    plot_imputation,
    _save_or_show
)

device  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
cfg     = VAEConfig()
sim_cfg = SimulationConfig()
tcfg    = TrainConfig()
os.makedirs('results/plots', exist_ok=True)
print(f'Device: {device}')

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams["font.family"] = "serif" 

## 2. Load Models

In [ ]:
ROOT = os.path.abspath('..')

CHECKPOINTS = {
    'KVAE':    os.path.join(ROOT, 'checkpoints/kvae/best_kvae.pt'),
    'GRU-VAE': os.path.join(ROOT, 'checkpoints/gru_vae/best_gru_vae.pt'),
    'CV-VAE':  os.path.join(ROOT, 'checkpoints/cv_vae/best_cv_vae.pt'),
}

MODEL_TYPES = {
    'KVAE':    'kvae',
    'GRU-VAE': 'gru_vae',
    'CV-VAE':  'cv_vae',
}

models = {
    name: load_model(ckpt, MODEL_TYPES[name], cfg, sim_cfg, tcfg, device)
    for name, ckpt in CHECKPOINTS.items()
    if os.path.exists(ckpt)
}

print('Loaded models:', list(models.keys()))

## 3. Dataset

In [ ]:
test_dataset = BallDataset(sim_cfg, cfg, tcfg, split='test')
test_loader  = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=0)
print(f'Test samples: {len(test_dataset)}')

## 4. Quantitative Evaluation
MSE across prediction horizons, split by gravity condition.

In [ ]:
MAX_PRED_STEPS = int(sim_cfg.T * tcfg.max_mask_ratio)
SMOOTHER       = False   # set True to evaluate with RTS smoother

all_metrics      = {}
all_mse_no_grav  = {}
all_mse_grav     = {}

for name, model in models.items():
    print(f'Evaluating {name}...')
    smoother_flag = SMOOTHER and (name == 'KVAE')

    metrics = compute_metrics(model, test_loader, smoother=smoother_flag, device=device)
    mse_no_grav, mse_grav = compute_mse_per_step(
        model, test_loader, MAX_PRED_STEPS, smoother=smoother_flag, device=device)

    all_metrics[name]     = metrics
    all_mse_no_grav[name] = mse_no_grav
    all_mse_grav[name]    = mse_grav

    print(f'  recon_mse={metrics["recon_mse"]:.5f}  '
          f'smoothness={metrics["latent_smoothness"]:.5f}')

In [ ]:
# Build summary table
rows = []
for name, m in all_metrics.items():
    row = {'Model': name}
    row['Recon MSE'] = f"{m['recon_mse']:.5f}"
    for n in [1, 5, 10]:
        row[f'Pred-{n} (no grav)'] = f"{all_mse_no_grav[name][n-1]:.5f}"
        row[f'Pred-{n} (grav)']    = f"{all_mse_grav[name][n-1]:.5f}"
    rows.append(row)

df = pd.DataFrame(rows).set_index('Model')
display(df)
df.to_csv('results/metrics_table.csv')
print('Saved: results/metrics_table.csv')

### MSE vs Prediction Horizon

In [ ]:
COLORS = {'KVAE': 'seagreen', 'GRU-VAE': 'steelblue', 'CV-VAE': 'tomato'}
STYLES = {'KVAE': '-',        'GRU-VAE': '--',         'CV-VAE': ':'}
steps = np.arange(1, MAX_PRED_STEPS + 1)

fig, axes = plt.subplots(2, 1, figsize=(7, 9), sharex=True, dpi=150)

for i, (ax, title, mse_dict) in enumerate(zip(
    axes,
    ['(a) Without gravity', '(b) With gravity'],
    [all_mse_no_grav, all_mse_grav]
)):
    for name, mse in mse_dict.items():
        ax.plot(steps, mse, 
                label=name,
                color=COLORS[name], 
                linestyle=STYLES[name],
                linewidth=1.8, 
                marker='o', 
                markersize=4, 
                markerfacecolor='white', 
                markeredgewidth=1.2)
    
    ax.set_ylabel('Mean Squared Error (MSE)', fontsize=11)
    ax.set_title(title, fontsize=12, loc='left', fontweight='bold')
    ax.grid(True, linestyle='--', alpha=0.6)
    
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

axes[1].set_xlabel('Prediction horizon $n$ (steps)', fontsize=11)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, 0.98), 
           ncol=3, frameon=True, fontsize=10)

plt.tight_layout(rect=[0, 0.03, 1, 0.95])

plt.savefig('results/plots/mse_per_step.png', dpi=300, bbox_inches='tight')
plt.show()

## 5. Per-Sample Visualizations

In [ ]:
SAMPLE_IDX   = 0
N_FREE_STEPS = MAX_PRED_STEPS

sample = test_dataset[SAMPLE_IDX]
ball_seq_s, obstacle_img_s, *rest = sample
u_seq_s = rest[0] if rest else None

ball_seq_s     = ball_seq_s.unsqueeze(0).to(device)
obstacle_img_s = obstacle_img_s.unsqueeze(0).to(device)
if u_seq_s is not None:
    u_seq_s = u_seq_s.unsqueeze(0).to(device)

# mask last N_FREE_STEPS frames
T    = sim_cfg.T
mask = torch.ones(1, T, device=device)
start_idx = int(T // 2 - N_FREE_STEPS // 2)
end_idx   = int(start_idx + N_FREE_STEPS)
mask[:, start_idx:end_idx] = 0.0
mask_np = mask[0].cpu().numpy()

@torch.no_grad()
def run(model, smoother=False):
    kw = dict(u_seq=u_seq_s, mask=mask)
    if hasattr(model, 'kalman'):
        kw['smoother'] = smoother
    out = model(ball_seq_s, obstacle_img_s, **kw)
    (x_dist, a_dist, a_seq, a_smooth, a_pred,
     z_dist, z_smooth, z_pred, R, Q, alpha_seq) = out

    result = {
        'x_hat':     x_dist.mean[0].cpu().numpy(),
        'a_mu':      a_dist.loc[0].cpu().numpy(),
        'a_smooth':  a_smooth[0].cpu().numpy(),
        'a_pred':    a_pred[0].cpu().numpy(),
        'ball_seq':  ball_seq_s[0].cpu().numpy(),
        'alpha_seq': alpha_seq[0].cpu().numpy() if alpha_seq is not None else None,
    }
    if z_dist is not None:
        P = z_dist.covariance_matrix[0].cpu().numpy()   # [T, dz, dz]
        result['P_diag'] = np.diagonal(P, axis1=-2, axis2=-1)
    return result

# run all models (filter mode)
outputs = {name: run(model, smoother=False) for name, model in models.items()}

# run KVAE also with smoother for comparison
if 'KVAE' in models:
    outputs['KVAE (smoother)'] = run(models['KVAE'], smoother=True)

print('Forward passes done.')

### 5.1 Latent Space Trajectories

In [ ]:
TRAJ_COLORS = {'KVAE': 'seagreen', 'GRU-VAE': 'steelblue', 'CV-VAE': 'tomato'}

show_names = list(outputs.keys()) 

for name in show_names:
    file_safe_name = name.replace(' (smoother)', '_smoother').lower()
    
    fig, ax = plt.subplots(figsize=(8, 5))
    
    out = outputs[name]
    main_col = TRAJ_COLORS.get(name.split(' (')[0], 'darkorange')
    a_mu = out['a_mu']
    a_sm = out['a_smooth']

    ax.plot(a_mu[:, 0], a_mu[:, 1],
            color='gray', linestyle=':', linewidth=1.0, alpha=0.4, 
            label='Encoder $a_\\mu$', zorder=1)

    path_label = 'Inferred Trajectory' if 'smoother' not in name else 'Smoothed Trajectory'
    ax.plot(a_sm[:, 0], a_sm[:, 1],
            color=main_col, linewidth=2.5, label=path_label, zorder=2)

    free = a_sm[start_idx:end_idx]
    ax.plot(free[:, 0], free[:, 1],
            color='#FF4500', linewidth=2.5, linestyle='--',
            label='Free-running (Pred.)', zorder=3)

    step = max(1, len(a_sm) // 8)
    for i in range(0, len(a_sm) - 1, step):
        arrow_col = plt.cm.get_cmap('Reds')(0.8) if main_col == '#CD5C5C' else 'black'
        
        ax.annotate('', xy=(a_sm[i+1, 0], a_sm[i+1, 1]),
                    xytext=(a_sm[i, 0], a_sm[i, 1]),
                    arrowprops=dict(arrowstyle='->', color='black', alpha=0.5, lw=1.0),
                    zorder=2)

    ax.scatter(*a_sm[0],  color='#32CD32', s=120, zorder=5, label='Start $t=1$',
               edgecolors='black', linewidth=1.0)
    ax.scatter(*a_sm[-1], color='#DC143C', s=120, zorder=5, label='End $t=T$',
               marker='X', edgecolors='black', linewidth=1.0)

    ax.set_title(f'Latent Space: {name}', fontsize=14, fontweight='bold', pad=15)
    ax.set_xlabel('Dimension $a_1$', fontsize=12)
    ax.set_ylabel('Dimension $a_2$', fontsize=12)
    
    ax.grid(True, linestyle='--', alpha=0.5)
    ax.axis('equal')
    
    ax.legend(fontsize=8, loc='best', frameon=True, shadow=False)

    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    plt.tight_layout()
    
    save_path = f'results/plots/latent_traj_{file_safe_name}.png'
    plt.savefig(save_path, dpi=300, bbox_inches='tight')

### 5.2 Filter vs RTS Smoother (KVAE)

In [ ]:
if 'KVAE' in models and 'KVAE (smoother)' in outputs:
    out_f = outputs['KVAE']
    out_s = outputs['KVAE (smoother)']

    plot_imputation(
        a_mu     = out_f['a_mu'],
        a_filt   = out_f['a_smooth'],
        a_smooth = out_s['a_smooth'],
        mask     = mask_np,
        save_path= 'results/plots/filter_vs_smoother.png'
    )

    dim = out_f['a_mu'].shape[-1]
    fig, axes = plt.subplots(dim, 1, figsize=(10, 3.5 * dim), sharex=True, dpi=150)
    if dim == 1:
        axes = [axes]
    
    t = np.arange(T)
    
    for i, ax in enumerate(axes):
        ax.plot(t, out_f['a_mu'][:, i], color='#D3D3D3', lw=1.0, 
                linestyle=':', label='Encoder $a_\\mu$')
        
        ax.plot(t, out_f['a_smooth'][:, i], color="#21527A", lw=2.0, 
                label='Kalman Filter (Online)')
        
        ax.plot(t, out_s['a_smooth'][:, i], color="#71F2A9", lw=2.2, 
                label='RTS Smoother (Offline)')
        
        ax.axvspan(start_idx, end_idx - 1, facecolor='whitesmoke', alpha=0.9, 
                   edgecolor='none', label='Masked Gap (Imputation)')
        
        ax.set_ylabel(f'Latent Dim $a_{i+1}$', fontsize=11, fontweight='bold')
        ax.grid(True, linestyle='--', alpha=0.5)
        
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        
        if i == 0:
            ax.set_title('Filter vs. RTS Smoother: \nLatent Component Comparison', 
                         fontsize=13, fontweight='bold', pad=20, loc='left')
            ax.legend(loc='lower left', bbox_to_anchor=(0.5, 1.0), ncol=2, 
                      frameon=False, fontsize=8, columnspacing=1.0)
            
    plt.xlabel('Time step $t$', fontsize=11, fontweight='bold')

    plt.tight_layout(rect=[0, 0, 1, 0.93]) 
    
    plt.savefig('results/plots/filter_vs_smoother_components.png', dpi=300, bbox_inches='tight')
    plt.show()

### 5.3 Posterior Uncertainty (KVAE)

In [ ]:
if 'KVAE' in models and 'P_diag' in outputs['KVAE']:
    plot_uncertainty(
        P_diag    = outputs['KVAE']['P_diag'],
        mask      = mask_np,
        smoother  = False,
        save_path = 'results/plots/uncertainty_filter.png'
    )

if 'KVAE (smoother)' in outputs and 'P_diag' in outputs['KVAE (smoother)']:
    plot_uncertainty(
        P_diag    = outputs['KVAE (smoother)']['P_diag'],
        mask      = mask_np,
        smoother  = True,
        save_path = 'results/plots/uncertainty_smoother.png'
    )

### 5.4 Reconstruction Grid

In [ ]:
# compare all models on the same sample
ball_np = outputs[list(outputs.keys())[0]]['ball_seq']   # ground truth
T_full  = ball_np.shape[0]
frame_indices = [0, T_full // 4, T_full // 2, T_full - N_FREE_STEPS, T_full - 1]

n_models = len(models)
n_frames = len(frame_indices)
fig, axes = plt.subplots(n_models + 1, n_frames, figsize=(n_frames * 2.5, (n_models + 1) * 2.5))

row_labels = ['Original'] + list(models.keys())
row_data   = [ball_np] + [outputs[n]['x_hat'] for n in models]

for row_idx, (label, data) in enumerate(zip(row_labels, row_data)):
    for col_idx, t in enumerate(frame_indices):
        ax = axes[row_idx, col_idx]
        ax.imshow(data[t].squeeze(), cmap='gray', vmin=0, vmax=1)
        ax.axis('off')
        if col_idx == 0:
            ax.set_ylabel(label, fontsize=10, rotation=90, labelpad=40, va='center')
        if row_idx == 0:
            marker = ' *' if t >= start_idx and t < end_idx else ''
            axes[row_idx, col_idx].set_title(f't={t}{marker}', fontsize=9)

plt.suptitle('Reconstruction Grid  (* = free-running prediction)', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('results/plots/reconstruction_all.png', dpi=150, bbox_inches='tight')
plt.show()

### 5.5 Alpha Weights Over Time (KVAE)

In [ ]:
if 'KVAE' in outputs and outputs['KVAE']['alpha_seq'] is not None:
    plot_alpha(
    alpha_seq = outputs['KVAE']['alpha_seq'],
    ball_seq  = outputs['KVAE']['ball_seq'],
    obs_img   = obstacle_img_s[0].cpu().numpy(), 
    free_run_range = (start_idx, end_idx),
    save_path = 'results/plots/alpha_kvae.png'
)

## 6. Latent Space Structure
Systematic coverage of all ball positions to visualize the encoder mapping.

In [ ]:
grid_path = ('dataset/univers.npy')

if os.path.exists(grid_path):
    obs = torch.tensor(np.load(grid_path), dtype=torch.float32)
    obs = obs.unsqueeze(1).to(device)
    
    with torch.no_grad():
        a_grid = models['KVAE'].ball_encoder(obs / 255.0).mean.squeeze(1)
    a_grid = a_grid.cpu().numpy()

    fig, ax = plt.subplots(figsize=(8, 8), dpi=150)

    ax.plot(a_grid[:, 0], a_grid[:, 1], 
            color='black', lw=0.5, alpha=0.4, zorder=1)

    ax.scatter(a_grid[:, 0], a_grid[:, 1], 
               color='steelblue', s=10, alpha=0.6, zorder=2)

    ax.scatter(a_grid[0, 0], a_grid[0, 1], color='forestgreen', s=60, 
               edgecolors='black', zorder=3, label='Start (First grid cell)')
    ax.scatter(a_grid[-1, 0], a_grid[-1, 1], color='crimson', s=60, marker='X', 
               edgecolors='black', zorder=3, label='End (Last grid cell)')

    ax.set_xlabel('$a_1$', fontsize=12)
    ax.set_ylabel('$a_2$', fontsize=12)
    ax.set_title('Latent Space Topology: Sequential Path', fontsize=13, fontweight='bold')
    
    ax.legend(fontsize=9, frameon=False)
    ax.grid(True, linestyle=':', alpha=0.5)
    ax.set_facecolor('white')
    
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    plt.tight_layout()
    plt.savefig('results/plots/latent_space_simple_path.png', dpi=300, bbox_inches='tight')
    plt.show()

    print("Simple path plot finalized. Clean and minimalist.")

## 7. Multi-Sample Visualizations (KVAE)

In [ ]:
N_SAMPLES = 5

for i in range(N_SAMPLES):
    sample_i = test_dataset[i]
    ball_i, obs_i, *rest_i = sample_i
    u_i = rest_i[0] if rest_i else None

    ball_i = ball_i.unsqueeze(0).to(device)
    obs_i  = obs_i.unsqueeze(0).to(device)
    if u_i is not None:
        u_i = u_i.unsqueeze(0).to(device)

    mask_i = torch.ones(1, T, device=device)
    mask_i[:, T - N_FREE_STEPS:] = 0.0

    with torch.no_grad():
        out_i = models['KVAE'](ball_i, obs_i, u_seq=u_i, mask=mask_i, smoother=False)

    (x_dist_i, a_dist_i, a_seq_i, a_smooth_i, a_pred_i,
     z_dist_i, z_smooth_i, z_pred_i, R_i, Q_i, alpha_i) = out_i

    sample_dir = f'results/plots/sample_{i}'
    os.makedirs(sample_dir, exist_ok=True)

    plot_trajectories(
        a_mu      = a_dist_i.loc[0].cpu().numpy(),
        a_smooth  = a_smooth_i[0].cpu().numpy(),
        free_run_range = (start_idx, end_idx),
        save_path = f'{sample_dir}/trajectories.png'
    )

    plot_reconstruction_grid(
        ball_seq   = ball_i[0].cpu().numpy(),
        x_hat_filt = x_dist_i.mean[0].cpu().numpy(),
        save_path  = f'{sample_dir}/reconstruction.png'
    )

    if alpha_i is not None:
        plot_alpha(
            alpha_seq = alpha_i[0].cpu().numpy(),
            ball_seq  = ball_i[0].cpu().numpy(),
            obs_img   = obs_i.cpu().numpy(),
            save_path = f'{sample_dir}/alpha.png'
        )

    if z_dist_i is not None:
        P_i = z_dist_i.covariance_matrix[0].cpu().numpy()
        plot_uncertainty(
            P_diag    = np.diagonal(P_i, axis1=-2, axis2=-1),
            mask      = mask_i[0].cpu().numpy(),
            save_path = f'{sample_dir}/uncertainty.png'
        )

    print(f'Sample {i} saved to {sample_dir}')